# Belief Networks Analysis with Expert Consensus

This notebook implements a belief network analysis system where multiple experts classify images, and their certainty is represented as triangular fuzzy functions.

## Key Concepts:
- **Experts**: Individual classifiers who assign labels to images
- **Triangular Functions**: Represent uncertainty as (min, apex, max) values
- **Convolution**: When multiple experts agree on a label, we convolve their certainty functions
- **Consensus**: The final classification considers agreement between experts


This has some issues, and does not seem to handle multiple experts well.  
the triangles from BNI3f don't seem to be scaled properly either.  
This may or may not be related to the multiple expert case.

## 1. Import Libraries and Configuration

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from collections import defaultdict
from scipy.stats import triang

import time
import datetime
import warnings
warnings.filterwarnings('ignore')

# ============== CONFIGURATION ==============
# File paths
FILE_PATH = "data_files/dataset_50_1.dat"  # Input file path

# PCA settings
USE_PCA = True  # Whether to use PCA for dimensionality reduction
PCA_VARIANCE_RETAINED = 0.99  # Retain 99% of variance

# Simulation settings
NUM_SIMULATIONS = 1000  # Number of simulated annealing runs
NUM_BEST_POINTS = 300  # Number of best solutions to keep

# Simulated annealing parameters
SA_INITIAL_TEMP = 1e-3
SA_MIN_TEMP = 1e-8
SA_COOLING_RATE = 0.75
SA_EARLY_STOP_ROUNDS = 25_000
SA_EARLY_STOP_TOL = 1e-4


# Visualization settings
SHOW_PLOTS = True  # Set to False to save plots instead of displaying
DEBUG_MODE = False  # Set to True for detailed output

# Numerical settings
NUM_DISCRETIZATION_POINTS = 1250  # Points for discretizing triangular functions

## 2. Core Functions for Triangular Fuzzy Logic

In [2]:
def triangle_function(tri, num_points=NUM_DISCRETIZATION_POINTS):
    min_val, apex, max_val = tri
    
    # SciPy's triang uses a different parameterization
    # c = (mode - loc) / scale
    loc = min_val  # left bound
    scale = max_val - min_val  # width
    c = (apex - min_val) / (max_val - min_val)  # normalized mode position
    
    x = np.linspace(min_val, max_val, num_points)
    y = triang.pdf(x, c, loc=loc, scale=scale)
    
    return x, y


def convolve_triangles(triangle_list, num_points=NUM_DISCRETIZATION_POINTS):
    """
    Convolve multiple triangular functions to represent combined certainty.
    
    Args:
        triangle_list: List of triangles [(min, apex, max), ...]
        num_points: Number of discretization points
    
    Returns:
        x: Support of convolved function
        y: Convolved probability distribution
        mode_x: Mode (peak) of the distribution
    """
    if not triangle_list:
        return None, None, None
    
    if len(triangle_list) == 1:
        x, y = triangle_function(triangle_list[0], num_points=num_points)
        y = y / np.sum(y)  # Normalize
        mode_x = x[np.argmax(y)]
        return x, y, mode_x
    
    # Start with first triangle
    _, y = triangle_function(triangle_list[0], num_points=num_points)
    total_min = triangle_list[0][0]
    total_max = triangle_list[0][2]
    
    # Convolve with remaining triangles
    for tri in triangle_list[1:]:
        _, y2 = triangle_function(tri, num_points=num_points)
        y = np.convolve(y, y2, mode='full')
        total_min += tri[0]
        total_max += tri[2]
    
    # Normalize and create support
    y = y / np.sum(y)
#    x = np.linspace(total_min, total_max, len(y)) 
    x = np.linspace(total_min, total_max, len(y))

    mode_x = x[np.argmax(y)]
    
    return x, y, mode_x

## 3. Data Loading and Processing Functions

In [3]:
def parse_data_file(filepath):
    """
    Parse the belief network data file.
    
    File format:
    - Line 1: num_photos, num_possible_labels, num_entries
    - Line 2: 0 (ignored)
    - Line 3: b vector (target values)
    - Lines 4+: expert_id, photo_id, label, min, apex, max
    
    Returns:
        dict: Parsed data including expert ratings and metadata
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    # Parse header
    header = lines[0].split()
    num_photos = int(header[0])
    num_possible_labels = int(header[1])
    num_entries = int(header[2])
    
    # Parse b vector
    b_vector = np.array(lines[2].split()).astype(np.float64)
    
    # Parse expert ratings
    ratings = []
    expert_ids = set()
    
    for i in range(3, min(len(lines), num_entries + 3)):
        parts = lines[i].split()
        if len(parts) >= 6 and int(parts[0]) != 0:  # Valid entry
            expert_id = int(parts[0])
            photo_id = int(parts[1]) - 1  # Convert to 0-based
            label = int(parts[2]) - 1  # Convert to 0-based
            triangle = (float(parts[3]), float(parts[4]), float(parts[5]))
            
            ratings.append({
                'expert_id': expert_id,
                'photo_id': photo_id,
                'label': label,
                'triangle': triangle
            })
            expert_ids.add(expert_id)
    
    # Count actual number of experts
    num_experts = len(expert_ids)
    
    return {
        'num_photos': num_photos,
        'num_possible_labels': num_possible_labels,
        'num_experts': num_experts,
        'expert_ids': sorted(expert_ids),
        'b_vector': b_vector,
        'ratings': ratings
    }


def build_consensus_matrices(data):
    """
    Build A matrix and F matrix based on expert consensus.
    
    Rules:
    - If multiple experts assign same label to same photo: convolve their triangles
    - If experts assign different labels: divide by number of different labels
    
    Returns:
        A_matrix: Matrix of expected values
        F_matrix: Matrix of full distributions
    """
    num_photos = data['num_photos']
    num_labels = data['num_possible_labels']
    
    # Initialize matrices
    A_matrix = np.zeros((num_photos, num_labels))
    F_matrix = [[None for _ in range(num_labels)] for _ in range(num_photos)]
    
    # Group ratings by (photo_id, label)
    photo_label_ratings = defaultdict(list)
    photo_labels = defaultdict(set)
    
    for rating in data['ratings']:
        key = (rating['photo_id'], rating['label'])
        photo_label_ratings[key].append(rating['triangle'])
        photo_labels[rating['photo_id']].add(rating['label'])
    
    # Process each photo-label combination
    for (photo_id, label), triangles in photo_label_ratings.items():
        # Get number of different labels assigned to this photo
        num_different_labels = len(photo_labels[photo_id])
        
        # Convolve triangles for this photo-label combination
        x, y, mode_x = convolve_triangles(triangles)
        
        if mode_x is not None:
            # Apply normalization based on number of different labels
            normalized_mode = mode_x / num_different_labels
            A_matrix[photo_id, label] = normalized_mode
            
            # Store full distribution (also normalized)
            F_matrix[photo_id][label] = (x / num_different_labels, y)
    
    return A_matrix, F_matrix

## 4. Sampling and Optimization Functions

In [4]:
def sample_from_F_matrix(F_matrix):
    """
    Sample values from the probability distributions in F matrix.
    
    Args:
        F_matrix: Matrix of (x, y) distributions
    
    Returns:
        F_sampled: Matrix of sampled values
    """
    rows, cols = len(F_matrix), len(F_matrix[0]) if F_matrix else 0
    F_sampled = np.zeros((rows, cols))
    
    for i in range(rows):
        for j in range(cols):
            if F_matrix[i][j] is not None:
                x, y = F_matrix[i][j]
                # Sample from distribution
                sampled = np.random.choice(x, p=y / np.sum(y))
                F_sampled[i, j] = sampled
    
    return F_sampled


def simulated_annealing(A, b, early_stop_rounds=SA_EARLY_STOP_ROUNDS, 
                       early_stop_tol=SA_EARLY_STOP_TOL):
    """
    Solve for phi using simulated annealing to minimize ||b - A'φ||∞
    subject to: φ ≥ 0, Σφ = 1
    
    Args:
        A: Feature matrix (will be transposed internally)
        b: Target vector
        early_stop_rounds: Rounds without improvement before checking tolerance
        early_stop_tol: Tolerance for early stopping
    
    Returns:
        phi: Optimized weight vector
    """
    def calculate_error(phi):
        """Calculate error with penalties for constraints"""
        computed_b = A @ phi
        diff = b - computed_b
        
        # Primary objective: minimize sup norm (L∞ norm)
        sup_norm = np.max(np.abs(diff))
        
        # Constraint penalties
        negative_penalty = 100 * np.sum(np.abs(phi[phi < 0]))  # φ ≥ 0
        sum_constraint_penalty = 10 * (1 - np.sum(phi))**2    # Σφ = 1
        
        # Additional penalty for negative differences (b - A'φ < 0)
        #negative_diff_penalty = 100 * np.sum(np.abs(diff[diff < 0]))
        
        return sup_norm + negative_penalty + sum_constraint_penalty #+ negative_diff_penalty
    
    # Initialize
    n = A.shape[1]
    phi = np.zeros(n)
    current_error = calculate_error(phi)
    best_phi = phi.copy()
    best_error = current_error
    
    # Annealing parameters
    T = SA_INITIAL_TEMP
    T_min = SA_MIN_TEMP
    alpha = SA_COOLING_RATE
    step_size = 0.1
    
    # Early stopping tracking
    rounds_without_improvement = 0
    last_best_error = best_error
    
    inner_loop_iters = 40_000

    while T > T_min:
        for _ in range(inner_loop_iters):  # Inner loop iterations
            # Generate neighbor solution
            new_phi = phi.copy()
            idx = np.random.randint(n)
            new_phi[idx] += np.random.uniform(-step_size, step_size)
            new_phi = np.clip(new_phi, 0, None)  # Ensure non-negative
            
            # Skip if sum constraint severely violated
            if np.sum(new_phi) > 1:
                continue
            
            # Calculate new error
            new_error = calculate_error(new_phi)
            
            # Metropolis criterion
            delta = new_error - current_error
            if delta < 0 or np.random.random() < np.exp(-delta / T):
                phi = new_phi
                current_error = new_error
                
                if new_error < best_error:
                    best_error = new_error
                    best_phi = new_phi.copy()
                    rounds_without_improvement = 0
                else:
                    rounds_without_improvement += 1
            
            # Early stopping check
            if rounds_without_improvement > (early_stop_rounds):
                if abs(best_error - last_best_error) < early_stop_tol:
                    #print('stopped early')
                    #print(best_error)
                    return best_phi
                last_best_error = best_error
                rounds_without_improvement = 0
        
        # Cool down
        T *= alpha
        step_size *= alpha**0.5
    
    return best_phi

## 5. Main Analysis Pipeline

In [5]:
# Load and parse data
print("=" * 60)
print("Belief Network Analysis with Expert Consensus")
print("=" * 60)

print(f"\nLoading data from: {FILE_PATH}")
try:
    data = parse_data_file(FILE_PATH)
    print(f"\nData loaded successfully:")
    print(f"  - Number of photos: {data['num_photos']}")
    print(f"  - Number of possible labels: {data['num_possible_labels']}")
    print(f"  - Number of unique experts: {data['num_experts']}")
    print(f"  - Expert IDs: {data['expert_ids']}")
    print(f"  - Total ratings: {len(data['ratings'])}")
    print(f"  - Sum of b vector: {np.sum(data['b_vector']):.4f}")
except Exception as e:
    print(f"Error loading file: {e}")
    raise

# Build consensus matrices
print("\nBuilding consensus matrices...")
A_matrix, F_matrix = build_consensus_matrices(data)
print(f"A matrix shape: {A_matrix.shape}")
print(f"\nFirst 5 rows of A matrix:")
print(np.round(A_matrix[:5], 4))

# Transpose A for optimization (we solve A'φ = b)
A_transposed = A_matrix.T
b_vector = data['b_vector']

Belief Network Analysis with Expert Consensus

Loading data from: data_files/dataset_50_1.dat

Data loaded successfully:
  - Number of photos: 50
  - Number of possible labels: 10
  - Number of unique experts: 10
  - Expert IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  - Total ratings: 500
  - Sum of b vector: 1.0000

Building consensus matrices...
A matrix shape: (50, 10)

First 5 rows of A matrix:
[[0.     0.029  1.5108 0.     0.     0.     0.     0.0331 0.     0.    ]
 [0.     0.0539 0.     0.     0.1991 0.6049 0.     0.0643 0.     0.    ]
 [0.     0.     0.     2.0683 0.     0.     0.     0.     0.073  0.    ]
 [0.     0.     0.     0.     0.     0.     0.     3.6623 0.     0.    ]
 [0.     0.     0.     0.0877 0.     0.     0.     0.     0.     2.1793]]


## 6. Visualize F Matrix Distributions

In [6]:
# Plot F matrix histograms
print("\nPlotting F matrix distributions...")
rows = len(F_matrix)
cols = len(F_matrix[0]) if rows > 0 else 0

# Create figure with appropriate size
fig_height = max(8, rows * 1.5)
fig_width = max(10, cols * 2)
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(fig_width, fig_height))

# Ensure axes is always 2D
if rows == 1 and cols == 1:
    axes = np.array([[axes]])
elif rows == 1:
    axes = axes.reshape(1, -1)
elif cols == 1:
    axes = axes.reshape(-1, 1)

# Plot each distribution
for i in range(rows):
    for j in range(cols):
        if F_matrix[i][j] is not None:
            x, y = F_matrix[i][j]
            axes[i, j].hist(x, weights=y, bins=50, alpha=0.7, color='blue', edgecolor='black')
            #axes[i, j].bar(x, y, width=(x[1]-x[0]), alpha=0.7, color='blue', edgecolor='black')
            #axes[i, j].fill_between(x, y, alpha=0.7, color='blue', edgecolor='black')
            axes[i, j].set_xlim([0, 1])
            axes[i, j].set_title(f'Photo {i+1}, Label {j+1}', fontsize=8)
            axes[i, j].set_xlabel('Value', fontsize=6)
            axes[i, j].set_ylabel('Probability', fontsize=6)
            axes[i, j].tick_params(labelsize=6)
        else:
            axes[i, j].text(0.5, 0.5, 'No ratings', ha='center', va='center', 
                           transform=axes[i, j].transAxes, fontsize=8)
            axes[i, j].set_xlim([0, 1])

plt.suptitle('F Matrix: Consensus Probability Distributions', fontsize=12)
plt.tight_layout()

if SHOW_PLOTS:
    plt.show()
else:
    plt.savefig('f_matrix_distributions.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("F matrix distributions saved to f_matrix_distributions.png")


Plotting F matrix distributions...


KeyboardInterrupt: 

## 7. Test Sampling and Single Optimization Run

In [ ]:
# Sample from F matrix
print("\nSampling from F matrix distributions...")
F_sampled = sample_from_F_matrix(F_matrix)
print("Sample from F matrix (first 5 rows):")
print(np.round(F_sampled[:5], 4))

# Test single simulated annealing run
print("\nTesting single simulated annealing run...")
test_phi = simulated_annealing(A_transposed, b_vector)
test_error = b_vector - A_transposed @ test_phi
test_sup_norm = np.max(np.abs(test_error))

print(f"\nTest results:")
print(f"  - Sup norm (L∞): {test_sup_norm:.6f}")
print(f"  - Sum of φ: {np.sum(test_phi):.6f}")
print(f"  - Min φ value: {np.min(test_phi):.6f}")
print(f"  - Max φ value: {np.max(test_phi):.6f}")
print(f"  - Number of non-zero φ: {np.sum(test_phi > 1e-6)}")

## 8. Multiple Simulations for Robust Solutions

In [ ]:
# Run multiple simulations
print(f"\nRunning {NUM_SIMULATIONS} simulated annealing runs...")
print("This may take several minutes...\n")

all_solutions = []
all_sup_norms = []
start_time = time.time()

for i in range(NUM_SIMULATIONS):
    # Progress reporting
    if i % 10 == 0:
        elapsed = time.time() - start_time
        if i > 0:
            rate = i / elapsed
            remaining = (NUM_SIMULATIONS - i) / rate
            print(f"Progress: {i}/{NUM_SIMULATIONS} ({i/NUM_SIMULATIONS*100:.1f}%) - "
                  f"Est. remaining: {remaining:.1f}s")
        else:
            print(f"Progress: {i}/{NUM_SIMULATIONS} ({i/NUM_SIMULATIONS*100:.1f}%)")
    
    # Run optimization
    phi = simulated_annealing(A_transposed, b_vector)
    all_solutions.append(phi)
    
    # Calculate sup norm
    error = b_vector - A_transposed @ phi
    sup_norm = np.max(np.abs(error))
    all_sup_norms.append(sup_norm)

# Summary statistics
total_time = time.time() - start_time
print(f"\nCompleted in {total_time:.1f} seconds ({total_time/NUM_SIMULATIONS:.3f}s per simulation)")


In [ ]:

# Select best solutions
sorted_indices = np.argsort(all_sup_norms)
best_indices = sorted_indices[:NUM_BEST_POINTS]
best_solutions = np.array([all_solutions[i] for i in best_indices])
best_sup_norms = [all_sup_norms[i] for i in best_indices]

print(f"\nBest solutions summary:")
print(f"  - Best sup norm: {min(best_sup_norms):.6f}")
print(f"  - Worst of best sup norms: {max(best_sup_norms):.6f}")
print(f"  - Average of best sup norms: {np.mean(best_sup_norms):.6f}")
print(f"  - Std dev of best sup norms: {np.std(best_sup_norms):.6f}")

## 9. Visualize Solution Quality Distribution

In [ ]:
# Plot distribution of sup norms
plt.figure(figsize=(12, 5))

# All solutions
plt.subplot(1, 2, 1)
plt.hist(all_sup_norms, bins=50, alpha=0.7, color='blue', edgecolor='black')
plt.axvline(x=min(best_sup_norms), color='red', linestyle='--', 
            label=f'Best: {min(best_sup_norms):.6f}')
plt.axvline(x=np.mean(all_sup_norms), color='green', linestyle=':', 
            label=f'Mean: {np.mean(all_sup_norms):.6f}')
plt.xlabel('Sup Norm (L∞ Error)')
plt.ylabel('Frequency')
plt.title('Distribution of All Solutions')
plt.legend()
plt.grid(True, alpha=0.3)

# Best solutions
plt.subplot(1, 2, 2)
plt.hist(best_sup_norms, bins=30, alpha=0.7, color='green', edgecolor='black')
plt.axvline(x=min(best_sup_norms), color='red', linestyle='--', 
            label=f'Best: {min(best_sup_norms):.6f}')
plt.axvline(x=np.mean(best_sup_norms), color='blue', linestyle=':', 
            label=f'Mean: {np.mean(best_sup_norms):.6f}')
plt.xlabel('Sup Norm (L∞ Error)')
plt.ylabel('Frequency')
plt.title(f'Distribution of Best {NUM_BEST_POINTS} Solutions')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
if SHOW_PLOTS:
    plt.show()
else:
    plt.savefig('sup_norm_distributions.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Sup norm distributions saved to sup_norm_distributions.png")

## 10. Dimensionality Reduction with PCA (Optional)

In [ ]:
# Apply PCA if requested
if USE_PCA and best_solutions.shape[1] > 1:
    print(f"\nApplying PCA to retain {PCA_VARIANCE_RETAINED*100}% variance...")
    
    pca = PCA(n_components=PCA_VARIANCE_RETAINED)
    best_solutions_pca = pca.fit_transform(best_solutions)
    
    print(f"\nPCA Results:")
    print(f"  - Original dimensions: {best_solutions.shape[1]}")
    print(f"  - Reduced dimensions: {best_solutions_pca.shape[1]}")
    print(f"  - Explained variance ratios: {np.round(pca.explained_variance_ratio_, 4)}")
    print(f"  - Total variance explained: {np.sum(pca.explained_variance_ratio_):.4f}")
    
    # Plot explained variance
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.bar(range(1, len(pca.explained_variance_ratio_) + 1), 
            pca.explained_variance_ratio_)
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance Ratio')
    plt.title('Variance Explained by Each PC')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(range(1, len(pca.explained_variance_ratio_) + 1),
             np.cumsum(pca.explained_variance_ratio_), 'bo-')
    plt.axhline(y=PCA_VARIANCE_RETAINED, color='r', linestyle='--', 
                label=f'Target: {PCA_VARIANCE_RETAINED*100}%')
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Variance Explained')
    plt.title('Cumulative Variance Explained')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.savefig('pca_analysis.png', dpi=150, bbox_inches='tight')
        plt.close()
        print("PCA analysis saved to pca_analysis.png")
    
    data_for_clustering = best_solutions_pca
else:
    print("\nSkipping PCA (not requested or insufficient dimensions)")
    data_for_clustering = best_solutions

## 11. Cluster Analysis of Solutions

In [ ]:
# Determine optimal number of clusters
print("\nPerforming cluster analysis...")
max_k = min(20, len(best_solutions) // 2)
k_range = range(2, max_k + 1)

inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data_for_clustering)
    inertias.append(kmeans.inertia_)
    
    if k < len(data_for_clustering):
        score = silhouette_score(data_for_clustering, kmeans.labels_)
        silhouette_scores.append(score)

# Find optimal k using elbow method and silhouette score
differences = np.diff(inertias)


differences_2nd = np.diff(differences)
elbow_k = np.argmax(differences_2nd) + 3  # +3 because we start at k=2 and take 2nd derivative

# Alternative: use percentage change
percentage_changes = np.abs(np.diff(inertias) / inertias[:-1]) * 100
threshold_indices = np.where(percentage_changes < 10)[0]  # Where improvement is less than 10%
threshold_k = threshold_indices[0] + 2 if len(threshold_indices) > 0 else elbow_k

# Choose optimal k
suggested_k = max(min(elbow_k, threshold_k, 10), 3)  # Between 3 and 10 clusters

print(f"\nOptimal cluster analysis:")
print(f"  - Elbow method suggests: {elbow_k} clusters")
print(f"  - Threshold method suggests: {threshold_k} clusters")
print(f"  - Selected: {suggested_k} clusters")


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.ndimage import gaussian_filter1d

print("\nPerforming cluster analysis...")
max_k = min(20, len(best_solutions) // 2)
k_range = np.arange(2, max_k + 1)

inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(data_for_clustering)
    inertias.append(kmeans.inertia_)
    if k < len(data_for_clustering):
        score = silhouette_score(data_for_clustering, kmeans.labels_)
        silhouette_scores.append(score)

inertias = np.array(inertias)

# --- Method 1: First derivative threshold ---
diffs = np.diff(inertias)
percent_changes = -diffs / inertias[:-1] * 100  # negative for percent decrease
thresh1 = 10  # percent threshold for diminishing returns
indices1 = np.where(percent_changes < thresh1)[0]
k_1 = k_range[indices1[0]] if len(indices1) > 0 else k_range[-1]

# --- Method 2: Smoothed first derivative threshold ---
smooth_inertias = gaussian_filter1d(inertias, sigma=1)
smooth_diffs = np.diff(smooth_inertias)
smooth_percent_changes = -smooth_diffs / smooth_inertias[:-1] * 100
indices2 = np.where(smooth_percent_changes < thresh1)[0]
k_2 = k_range[indices2[0]] if len(indices2) > 0 else k_range[-1]

# --- Method 3: Relative to the initial drop ---
initial_change = -((inertias[1] - inertias[0]) / inertias[0]) * 100
rel_threshold = initial_change / 2  # e.g. half the first drop
indices3 = np.where(percent_changes < rel_threshold)[0]
k_3 = k_range[indices3[0]] if len(indices3) > 0 else k_range[-1]

# --- Final selection (choose the median for robust choice) ---
all_ks = np.array([k_1, k_2, k_3])
suggested_k = int(np.median(all_ks))

print(f"\nCluster analysis with 3 simple elbow methods:")
print(f"  - First derivative threshold (<{thresh1}%): {k_1} clusters")
print(f"  - Smoothed first-derivative (<{thresh1}%): {k_2} clusters")
print(f"  - Less than half the initial improvement: {k_3} clusters")
print(f"  - Final suggested number of clusters: {suggested_k}\n")

In [ ]:
plt.figure(figsize=(6, 5))

plt.plot(k_range, inertias, 'bo-', linewidth=2)
plt.axvline(x=suggested_k, color='red', linestyle='--',
            label=f'Selected k={suggested_k}')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.title('Elbow Method for Optimal k')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
if SHOW_PLOTS:
    plt.show()
else:
    plt.savefig('clustering_analysis.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Clustering analysis saved to clustering_analysis.png")

## 12. Final Clustering and Analysis

In [ ]:
# Perform final clustering
print(f"\nPerforming final clustering with k={suggested_k}...")
final_kmeans = KMeans(n_clusters=suggested_k, random_state=42, n_init=20)
cluster_labels = final_kmeans.fit_predict(data_for_clustering)

# Calculate cluster centroids in original space
cluster_centroids = []
cluster_info = []

for k in range(suggested_k):
    cluster_points = best_solutions[cluster_labels == k]
    centroid = np.mean(cluster_points, axis=0)
    cluster_centroids.append(centroid)
    
    # Calculate statistics for this cluster
    centroid_error = b_vector - A_transposed @ centroid
    centroid_sup_norm = np.max(np.abs(centroid_error))
    
    cluster_sup_norms = [best_sup_norms[i] for i in range(len(best_sup_norms)) 
                        if cluster_labels[i] == k]
    
    cluster_info.append({
        'cluster_id': k,
        'n_points': len(cluster_points),
        'centroid_sup_norm': centroid_sup_norm,
        'mean_sup_norm': np.mean(cluster_sup_norms),
        'std_sup_norm': np.std(cluster_sup_norms),
        'min_sup_norm': np.min(cluster_sup_norms),
        'max_sup_norm': np.max(cluster_sup_norms)
    })

# Display cluster information
print("\nCluster Analysis Results:")
print("-" * 80)
for info in cluster_info:
    print(f"Cluster {info['cluster_id']}:")
    print(f"  - Points: {info['n_points']}")
    print(f"  - Centroid sup norm: {info['centroid_sup_norm']:.6f}")
    print(f"  - Mean sup norm: {info['mean_sup_norm']:.6f} ± {info['std_sup_norm']:.6f}")
    print(f"  - Range: [{info['min_sup_norm']:.6f}, {info['max_sup_norm']:.6f}]")
    print()

## 13. Save Results

In [ ]:
# Save results to files
print("\nSaving results...")
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Save all best solutions
with open('best_solutions.txt', 'w') as f:
    f.write(f"# Belief Network Analysis Results\n")
    f.write(f"# Generated on: {timestamp}\n")
    f.write(f"# Input file: {FILE_PATH}\n")
    f.write(f"# Number of photos: {data['num_photos']}\n")
    f.write(f"# Number of labels: {data['num_possible_labels']}\n")
    f.write(f"# Number of experts: {data['num_experts']}\n")
    f.write(f"# Best {NUM_BEST_POINTS} solutions from {NUM_SIMULATIONS} runs\n")
    f.write(f"# PCA used: {USE_PCA} (variance retained: {PCA_VARIANCE_RETAINED*100}%)\n")
    f.write(f"# Format: {best_solutions.shape[1]} values per line\n")
    f.write("#\n")
    for i, (solution, sup_norm) in enumerate(zip(best_solutions, best_sup_norms)):
        f.write(f"# Solution {i}: sup_norm = {sup_norm:.6f}\n")
        f.write(' '.join([f'{val:.6f}' for val in solution]) + '\n')

# Save cluster centroids
with open('cluster_centroids.txt', 'w') as f:
    f.write(f"# Cluster Centroids from Belief Network Analysis\n")
    f.write(f"# Generated on: {timestamp}\n")
    f.write(f"# Input file: {FILE_PATH}\n")
    f.write(f"# Number of clusters: {suggested_k}\n")
    f.write(f"# Format: {len(cluster_centroids[0])} values per line\n")
    f.write("#\n")
    for info, centroid in zip(cluster_info, cluster_centroids):
        f.write(f"# Cluster {info['cluster_id']}: {info['n_points']} points, ")
        f.write(f"sup_norm = {info['centroid_sup_norm']:.6f}\n")
        f.write(' '.join([f'{val:.6f}' for val in centroid]) + '\n')

# Save detailed analysis report
with open('analysis_report.txt', 'w') as f:
    f.write(f"Belief Network Analysis Report\n")
    f.write(f"="*50 + "\n\n")
    f.write(f"Generated on: {timestamp}\n")
    f.write(f"Input file: {FILE_PATH}\n\n")
    
    f.write(f"Data Summary:\n")
    f.write(f"  - Photos: {data['num_photos']}\n")
    f.write(f"  - Possible labels: {data['num_possible_labels']}\n")
    f.write(f"  - Unique experts: {data['num_experts']}\n")
    f.write(f"  - Total ratings: {len(data['ratings'])}\n\n")
    
    f.write(f"Optimization Summary:\n")
    f.write(f"  - Total simulations: {NUM_SIMULATIONS}\n")
    f.write(f"  - Best solutions kept: {NUM_BEST_POINTS}\n")
    f.write(f"  - Best sup norm achieved: {min(best_sup_norms):.6f}\n")
    f.write(f"  - Average of best sup norms: {np.mean(best_sup_norms):.6f}\n\n")
    
    if USE_PCA:
        f.write(f"PCA Summary:\n")
        f.write(f"  - Original dimensions: {best_solutions.shape[1]}\n")
        f.write(f"  - Reduced dimensions: {best_solutions_pca.shape[1]}\n")
        f.write(f"  - Variance retained: {np.sum(pca.explained_variance_ratio_):.4f}\n\n")
    
    f.write(f"Clustering Summary:\n")
    f.write(f"  - Number of clusters: {suggested_k}\n")
    for info in cluster_info:
        f.write(f"  - Cluster {info['cluster_id']}: {info['n_points']} points, ")
        f.write(f"centroid sup norm = {info['centroid_sup_norm']:.6f}\n")

print(f"\nResults saved to:")
print(f"  - best_solutions.txt: All {NUM_BEST_POINTS} best solutions")
print(f"  - cluster_centroids.txt: {suggested_k} cluster centroids")
print(f"  - analysis_report.txt: Detailed analysis summary")
print("\nAnalysis complete!")